# Deep Learning Detection of Exoplanet Transits in PSLS-Simulated PLATO Light Curves
## A Comparison with Classical Box Least Squares

Silvia Parente. 
University of Belgrade, Computational Astrobiology final project. March-June 2026

#### The Photometric Transit Method

The transit method is an indirect detection technique based on high-precision time-series photometry. A transit occurs when an exoplanet’s orbital plane is aligned with the observer’s line of sight, so that the planet's disk hides a portion of the star surface.

The morphology of a transit light curve provides a set of physical constraints like the transit duration ($T_{dur}$), ingress and egress (the slopes of the entry and exit points of the transit), the orbital period ($P$).

From an astrobiological perspective, the transit method is useful to identifying habitable zone candidates. It allows for the calculation of the planetary equilibrium temperature ($T_{eq}$) and identifies targets for future transmission spectroscopy. During a transit, the starlight filtering through the planetary annulus carries the absorption signatures of the chemical species in the atmosphere, potentially revealing biosignatures such as $O_2$, $O_3$, $CH_4$, and $H_2O$.



#### PLATO

The PLAnetary Transits and Oscillations of stars (PLATO) mission is a planned European Space Agency space telescope set to launch in 2026, with the primary mission target being the search for terrestrial exoplanets in the habitable zones of Sun-like stars using the transit method. The goal is to provide accurate key planet parameters like radius, mass, density and age in statistical numbers to help understand better how planetary systems form and evolve, or if there are other systems with planets like ours, including potentially habitable planets. 


noise?
One of the main specificities of the mission is that it relies on a multi-telescope concept. Among the 26 cameras that compose the instrument, two of them are named “fast” cameras and work at a 2.5 s cadence while the remaining 24 are named “normal” cameras and work at a 25 s cadence. The normal cameras are divided into four groups of six cameras, with large fields of view (about 1100 square degrees) that partially overlap. 
Because of the large field of view and the long-term change of the pointing direction of each individual camera, star positions will slowly drift on the camera focal plane during the 3-month uninterrupted observation sequences. As a consequence, stars will slowly leave the aperture photometry, leading to a long-term decrease of their measured intensities. Furthermore, during the life of the mission, the instrument will be continuously exposed to radiation (mostly proton impacts). 
To mitigate the flux variations induced by the instrument and the observational conditions, the aperture masks used on-board will be updated on a quasi-regular basis; this will leave residual flux variations that will be corrected later on-ground on the basis of the knowledge of the instrument, even if this correction will leave systematic errors in the power spectrum that will rapidly increase with decreasing frequency. 
All of these instrumental systematic errors together with the stellar activity noise component can impact the detection and characterisation of the planetary transits, limit the seismic analysis of very evolved red giant stars, and affect any science analysis of the signal at rather low frequencies.



#### PSLS
The preparation of science objectives will require the implementation of hare-and-hound exercises relying on the massive generation of representative simulated light-curves. The PLATO Solar-like Light-curve Simulator (PSLS) is a light-curve simulator that generates light-curves representative of typical PLATO targets, showing simultaneously solar-like oscillations, stellar granulation, and magnetic activity. At the same time, PSLS also aims at mimicking in a realistic way the random noise and the systematic errors representative of the PLATO multi-telescope concept.

The selection of PSLS as the primary simulation engine is dictated by its representation of the specific photometric challenges inherent to the PLATO mission. It implements a physically motivated model of solar-like oscillators and accounts for the multi-telescope architecture of PLATO by simulating instrumental systematics, including pointing jitter, CCD red noise, and discrete time shifts between camera groups. 


#### The Box Least Squares baseline
The Box Least Squares (BLS) algorithm is adopted as the classical benchmark due to its status as the statistically optimal frequentist approach for detecting periodic, step-function-like signals in time-series data. BLS operates by partitioning the folded light curve into two discrete states—in-transit and out-of-transit—and maximizing a signal-to-pink-noise statistic over a predefined grid of periods ($P$) and transit durations ($\Delta \theta$). 


### Installation of PSLS

The source package psls-1.9.tar.gz was downloaded from the official PSLS website. The package was extracted into the directory: /Users/silvi1/computational_astrobiology/psls-1.9. The installation was performed within a dedicated virtual environment named psls_env, initialized using python3 -m venv, accesed with source plato_env/bin/activate. Key libraries were installed via pip, including:
psls, astropy, matplotlib and numpy, pyyaml. 

PSLS was tested using the example configuration file 0009882316.yaml (a red giant star template) located in the examples/ subdirectory. The configuration file was modified to match the version requirements of PSLS 1.9. The following sections were updated:
- Granulation: added Type: 1 
- Observation: added Gaps: 0 

These modifications were necessary because the original example files lacked fields expected by the current version of PSLS.
Filled these lacks and finally obtained a lightcurve and plots for example 0009882316. \
The plot shows the Power Spectral Density (PSD) of the simulated star. It displays the oscillation modes of the simulated red giant star. \

The simulation ran correctly:

- A 730-day observation was simulated.

- The noise levels and systematic errors were properly applied.

- The oscillation modes match what is expected for a red giant star.


### Simulation grid

Here is the range of simulated physical and observational parameters.

| Parameter            | Numerical Range   | Reason for Choice                                                             |
| -------------------- | ----------------- | ----------------------------------------------------------------------------- |
| Orbital Period (P)   | 2.0 to 20.0 days  | Covers "Hot" to "Warm" planets; ensures multiple transits in a 30-day window. |
| Planet Radius (Rp​)  | 0.5 to 2.0 RJup​  | Varies the transit depth.         |
| Impact Parameter (b) | 0.0 to 0.8        | Varies the transit duration and shape (grazing vs. centered).                 |
| Transit Duration     | Dependent         | This changes automatically based on P and b.                  |
| White Noise          | 10 to 100 ppm     | Simulates different PLATO camera efficiencies and target brightness.          |
| Stellar Variability  | 1.0 to 5.0× Solar | Tests the classifier's ability to "see through" granulation and oscillations. |
| Random Seed          | 1 to 300          | Ensures every light curve has unique stochastic noise.                        |
| Stellar Mass (M∗​)   | 0.8 to 1.2 M⊙​    | Changes the transit duration and depth for the same planet size               |
| Stellar Age/Activity | 1.0 to 10.0 Gyr   | Older stars are "quieter", younger stars make detection much harder.          |


Distribution of Light Curves (N=300):

| regime | count | description |
| -------------------- | ----------------- | ----------------------------------------------------------------------------- |
| Easy (Deep) | 100 | Large planets ($R_p > 1.5 R_{Jup}$) + quiet stars |
| Intermediate | 100 | Medium planets ($R_p \\approx 1.0 R_{Jup}$) + average noise. |
| Difficult (Shallow) | 100 | Small planets ($R_p < 0.7 R_{Jup}$) + high noise/variability. |


In [ ]:
import numpy as np
import yaml
import os
import csv

N_TOTAL = 1000
csv_filename = "metadata_summary.csv"

header = ['unique_id', 'label', 'injected_period', 'injected_radius', 'noise_setting', 'duration', 'random_seed']
print(f"Starting generation of {N_TOTAL} light curves.")


with open(csv_filename, mode='w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(header)

    for i in range(1, N_TOTAL + 1):
        # 1 for Planet, 0 for No Planet
        has_planet = 1 if i <= 500 else 0
        
        # Difficulty settings (Radius in Jupiter Radii, Noise in ppm/hr)
        if i%3 == 0:
            #easy
            radius = np.random.uniform(1.5, 2.0)  
            noise = np.random.uniform(10, 30)
        elif i%3== 1:
            #medium
            radius = np.random.uniform(0.8, 1.2) 
            noise = np.random.uniform(40, 60)
        else:
            #hard
            radius = np.random.uniform(0.5, 0.7) 
            noise = np.random.uniform(70, 100)

        period = float(np.random.uniform(2.0, 20.0))
        duration = 30.0
        unique_id = f"{i:010d}"

        writer.writerow([unique_id, has_planet, period, radius, noise, duration, i])

        # 3. Create the YAML config
        config = {
            'Observation': {
                'Duration': duration,
                'QuarterDuration': [90.0],
                'MasterSeed': i,
                'Gaps': {'Enable': 0}
            },
            'Instrument': {
                'Sampling': 25.0,
                'IntegrationTime': 22.0,
                'GroupID': [1,1,1,1],
                'NGroup': 4,
                'NCamera': 6,
                'TimeShift': 6.25,
                'RandomNoise': {'Enable': 1, 'Type': 'PLATO_SCALING', 'NSR': float(noise)},
                'Systematics': {'Enable': 0, 'Version': 2, 'DriftLevel': 'low'}
            },
            'Star': {
                'Mag': 10.5,
                'ID': i,
                'ModelDir': '/Users/silvi1/plato_env/lib/python3.13/site-packages/psls/data/',
                'ModelType': 'UP',
                'ModelName': 'Sun',
                'ES': 'ms',
                'Teff': 5778.0,
                'Logg': 4.44,
                'SurfaceRotationPeriod': 25.0,
                'CoreRotationFreq': 0.0,
                'Inclination': 0.0,
                'Seed': -1
            },
            'Oscillations': {
                'Enable': 1, 'Seed': -1, 'numax': 3090.0, 'delta_nu': 135.0, 'DPI': -1, 'q': 0.7, 'SurfaceEffects': 0
            },
            'Activity': {'Enable': 0, 'Sigma': 1000.0, 'Tau': 30.0, 'Flare': {'Enable': 0}, 'Spot': {'Enable': 0}},
            'Granulation': {'Enable': 1, 'Type': 1, 'Seed': -1},
            'Transit': {
                'Enable': has_planet,
                'PlanetRadius': float(radius),
                'OrbitalPeriod': period,
                'PlanetSemiMajorAxis': 0.1,
                'OrbitalAngle': 0.0,
                'LimbDarkeningCoefficients': [0.25, 0.75]
            },
            'External': {'Enable': 0}
        }

        yaml_name = f"config_{i}.yaml"
        with open(yaml_name, 'w') as f:
            yaml.dump(config, f)

       
    
        print(f"--- Generating Case {i}/{N_TOTAL} (Has Planet: {has_planet}) ---")
        os.system(f"export PYTHONPATH=/Users/silvi1/plato_env/lib/python3.13/site-packages; echo '' | /Users/silvi1/plato_env/bin/python3 psls.py -V {yaml_name}")
        modes_file = f"{unique_id}.modes"
        if os.path.exists(modes_file):
            os.remove(modes_file)

print(f"Finished! Check your folder for {N_TOTAL} .dat files and {csv_filename}.")



### Safety Check

This generated the first 690 light curves. I wanted to check if it worked before starting a 10,000 run to avoid realizing that maybe the "hard" planets are actually invisible or the noise is so loud that even a human can't see the transit.
I want to test first if the difficulty levels I set (Easy, Medium, Hard) are actually working and confirm the noise distributions match what I expect before scaling up.

In [ ]:
import pandas as pd
import os

df = pd.read_csv('metadata_summary.csv')

# how many files actually exist
existing_files = [f for f in os.listdir('.') if f.endswith('.dat')]
print(f"Total .dat files found: {len(existing_files)}")

# balance check
balance = df[df['unique_id'].astype(int).isin([int(f.split('.')[0]) for f in existing_files])]
print("\nDataset Balance:")
print(balance['label'].value_counts(normalize=True)) 
print(balance['label'].value_counts())

In [ ]:
planets = df[df['label'] == 1]
print(f"--- Final Dataset Stats (N={len(existing_files)}) ---")
print(f"Shortest Period: {planets['injected_period'].min():.2f} days")
print(f"Longest Period: {planets['injected_period'].max():.2f} days")
print(f"Smallest Planet (Hard): {planets['injected_radius'].min():.2f} R_earth")
print(f"Largest Planet (Easy): {planets['injected_radius'].max():.2f} R_earth")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the metadata 
df = pd.read_csv('metadata_summary.csv')

# Create a 1x3 grid for distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['injected_period'], bins=20, ax=axes[0], color='skyblue')
axes[0].set_title('Distribution of Injected Periods')

sns.histplot(df['injected_radius'], bins=20, ax=axes[1], color='salmon')
axes[1].set_title('Distribution of Injected Depths (Radius)')

sns.histplot(df['noise_setting'], bins=20, ax=axes[2], color='green')
axes[2].set_title('Distribution of Noise (NSR)')

plt.tight_layout()
plt.show()

- Left Plot (Injected Periods): This shows how many days occur between transits. It’s a uniform distribution, so a fair mix of short-period planets (planets very close to the star) and long-period planets was successfully generated.

- Middle Plot (Injected Radius): three distinct humps show the three difficulty levels I programmed.

  - Smallest hump (0.5–0.7): hard cases, Earth-sized or smaller.
  - Middle hump (0.8–1.2): medium, Neptune-sized.
  - Largest hump (1.5–2.0): easy, Jupiter-sized.

- Right Plot (Noise Setting): noise to signal ratio. 
  - Low Noise 
  - Medium Noise 
  - High Noise 

In [ ]:
def plot_lc_final_zoom(id_value, title, ax):
    clean_id = f"{int(float(id_value)):010d}"
    filename = f"{clean_id}.dat"
    
    if os.path.exists(filename):
        data = np.loadtxt(filename)
        time_days = data[:, 0] / 86400 
        flux = data[:, 1]
        
        # Apply smoothing to reveal the trend
        window = 150
        smoothed = np.convolve(flux, np.ones(window)/window, mode='valid')
        
        # Plot smoothed data
        ax.plot(time_days[:len(smoothed)], smoothed, color='blue', lw=1)
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel("Time (Days)")
        ax.set_ylabel("Flux (ppm)")
        
        # --- THE KEY FIX: INDEPENDENT AUTO-ZOOM ---
        # This forces the Y-axis to fit ONLY the data in this specific plot
        y_min, y_max = np.min(smoothed), np.max(smoothed)
        margin = (y_max - y_min) * 0.1
        ax.set_ylim(y_min - margin, y_max + margin)
        # ------------------------------------------
        
    else:
        ax.set_title(f"Missing: {filename}")

# Create the 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Select specific examples from your new 901 files
easy = df[df['injected_radius'] > 1.8].iloc[0]
hard = df[(df['label'] == 1) & (df['injected_radius'] < 0.6)].iloc[0]
quiet = df[(df['label'] == 0) & (df['noise_setting'] < 30)].iloc[0]
active = df[(df['label'] == 0) & (df['noise_setting'] > 80)].iloc[0]

plot_lc_final_zoom(easy['unique_id'], f"EASY: Big Planet (Radius {easy['injected_radius']:.2f})", axes[0,0])
plot_lc_final_zoom(hard['unique_id'], f"HARD: Tiny Planet (Radius {hard['injected_radius']:.2f})", axes[0,1])
plot_lc_final_zoom(quiet['unique_id'], f"NEGATIVE: Quiet Star (NSR {quiet['noise_setting']:.2f})", axes[1,0])
plot_lc_final_zoom(active['unique_id'], f"NEGATIVE: Active Star (NSR {active['noise_setting']:.2f})", axes[1,1])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Case 1
data = np.loadtxt('0000000001.dat')
time_days = data[:, 0] / 86400  # Convert seconds to days
flux = data[:, 1]

fig, (ax2) = plt.subplots(1, 1, figsize=(12, 8))

ax2.plot(time_days, flux, lw=1)
ax2.set_xlim(1, 30) 
ax2.set_title("Zoomed View: Day 1 to Day 30")
ax2.set_xlabel("Time (Days)")
ax2.set_ylabel("Flux")

plt.tight_layout()
plt.show()